In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/data BE/"

print("BASE_DIR =", BASE_DIR)
print("Ce qu'il y a dans ce dossier :")
print(os.listdir(BASE_DIR))



In [ ]:
import json

def load_corpus(file_path):
    """
    Charge le corpus depuis un fichier JSONL.
    Retourne un dict : {doc_id: document}
    """
    corpus = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            corpus[doc["_id"]] = doc
    return corpus


def load_queries(file_path):
    """
    Charge les requêtes depuis un fichier JSONL.
    Retourne un dict : {query_id: document}
    """
    queries = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            queries[doc["_id"]] = doc
    return queries


def load_qrels(file_path):
    """
    Charge les jugements de pertinence depuis un fichier TSV.
    Gère automatiquement la présence d'un en-tête.
    Retourne : {query_id: {doc_id: relevance}}
    """
    qrels = {}
    with open(file_path, "r", encoding="utf-8") as f:
        first_line = True
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")

            # Gestion de l'en-tête
            if first_line:
                first_line = False
                if not parts[-1].isdigit():
                    continue  # on saute l'en-tête

            qid, did, rel = parts
            if qid not in qrels:
                qrels[qid] = {}
            qrels[qid][did] = int(rel)

    return qrels


In [ ]:
# Charger depuis Google Drive
corpus = load_corpus(BASE_DIR + "corpus.jsonl")
queries = load_queries(BASE_DIR + "queries.jsonl")
qrels_valid = load_qrels(BASE_DIR + "valid.tsv")


print("Nb docs dans le corpus :", len(corpus))
print("Nb de requêtes        :", len(queries))
print("Nb de requêtes avec qrels (valid.tsv) :", len(qrels_valid))



In [ ]:
!pip install sentence-transformers


In [ ]:
!pip install faiss-cpu


In [ ]:
def load_corpus_colab(file_path):
    corpus = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            doc = json.loads(line)
            doc_id = (
                doc.get("id")
                or doc.get("_id")
                or doc.get("doc_id")
                or doc.get("paper_id")
            )
            corpus[doc_id] = doc
    return corpus



In [ ]:
# Liste des IDs de documents
doc_ids = list(corpus.keys())

# Texte associé à chaque document (titre ou text)
doc_texts = [
    corpus[did].get("title") or corpus[did].get("text") or ""
    for did in doc_ids
]

print("Nombre de documents :", len(doc_ids))
print("Exemple texte d'un doc :")
print(doc_ids[0], "->", doc_texts[0][:200], "...")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Modèle recommandé (rapide, performant)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encodage du corpus (avec barre de progression)
corpus_embeddings = model.encode(
    doc_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape embeddings :", corpus_embeddings.shape)


In [ ]:
import pickle
import numpy as np

# Sauvegarder les embeddings et la liste des IDs
np.save("corpus_embeddings.npy", corpus_embeddings)

with open("doc_ids.pkl", "wb") as f:
    pickle.dump(doc_ids, f)


In [ ]:
import csv
from collections import defaultdict

def load_qrels(file_path):
    """
    valid.tsv format:
    query-id \t corpus-id \t score
    """
    qrels = defaultdict(dict)
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)  #  sauter l'en-tête
        for row in reader:
            if not row:
                continue
            qid, did, rel = row[0], row[1], int(row[2])
            qrels[qid][did] = rel
    return qrels


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive"):
    for name in files:
        if name in ["corpus.jsonl", "queries.jsonl", "valid.tsv"]:
            print(os.path.join(root, name))


In [ ]:
#  Charger les requêtes et les qrels
queries = load_queries(BASE_DIR + "queries.jsonl")
qrels_valid = load_qrels(BASE_DIR + "valid.tsv")
#queries = load_queries("queries.jsonl")
#qrels_valid = load_qrels("valid.tsv")

print("Nb de requêtes :", len(queries))
print("Nb de requêtes avec jugements (qrels_valid) :", len(qrels_valid))

#  Vérifier un exemple concret
first_qid = next(iter(qrels_valid.keys()))
print("\nExemple de qid :", first_qid)

# texte de la requête
q_text = queries[first_qid].get("title") or queries[first_qid].get("text") or ""
print("Texte de la requête :", q_text)

# candidats + labels pertinents / non pertinents
print("\nCandidats pour cette requête :")
for did, rel in qrels_valid[first_qid].items():
    print(f"  doc_id = {did} | rel = {rel}")


In [ ]:
import json

def load_queries(file_path):
    """
    Charge les requêtes depuis queries.jsonl
    retourne: dict[qid] -> dict(données de la requête)
    """
    queries = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            doc = json.loads(line)
            qid = (
                doc.get("id")
                or doc.get("_id")
                or doc.get("doc_id")
                or doc.get("paper_id")
            )
            queries[qid] = doc
    return queries

# ---- CHARGEMENT DES DONNÉES ----

queries = load_queries(BASE_DIR + "queries.jsonl")
qrels_valid = load_qrels(BASE_DIR + "valid.tsv")
#queries = load_queries("queries.jsonl")
#qrels_valid = load_qrels("valid.tsv")

print("Nb docs dans le corpus :", len(corpus))
print("Nb de requêtes        :", len(queries))
print("Nb de requêtes avec qrels (valid.tsv) :", len(qrels_valid))

# Petit contrôle sur un exemple
example_qid = list(qrels_valid.keys())[0]
print("\nExemple qid :", example_qid)
print("Titre de la requête :", queries[example_qid].get("title"))
print("Quelques candidats (doc_id, score) :", list(qrels_valid[example_qid].items())[:5])


In [ ]:
# Mapping doc_id -> index dans le tableau des embeddings
doc_index = {doc_id: i for i, doc_id in enumerate(doc_ids)}

# Vérification simple
print("Nb d'entrées dans doc_index :", len(doc_index))
print("Exemple :", list(doc_index.items())[:3])


In [ ]:
test_doc_id = doc_ids[0]

print("Doc ID testé :", test_doc_id)
print("Index correspondant :", doc_index[test_doc_id])
print("Shape de l'embedding :", corpus_embeddings[doc_index[test_doc_id]].shape)


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np



model = SentenceTransformer("all-MiniLM-L6-v2")

#  On prend uniquement les requêtes qui ont des qrels dans valid.tsv
query_ids = list(qrels_valid.keys())

#  Pour chaque qid, on récupère un texte (ici le titre, sinon 'text')
query_texts = [
    queries[qid].get("title") or queries[qid].get("text") or ""
    for qid in query_ids
]

print("Nb de requêtes à encoder :", len(query_ids))
print("Exemple de requête :", query_ids[0], "->", query_texts[0])

#  Encodage des requêtes en embeddings denses
query_embeddings = model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape des query_embeddings :", query_embeddings.shape)


In [ ]:
#  Liste des qids présents dans valid.tsv

valid_qids = list(qrels_valid.keys())
print("Nb de requêtes dans valid.tsv :", len(valid_qids))

#  On récupère le texte de chaque requête (titre ou texte),

query_texts = []
for qid in valid_qids:
    q = queries.get(qid, {})
    txt = q.get("title") or q.get("text") or ""
    query_texts.append(txt)

print("Exemple :")
print(valid_qids[0], "->", query_texts[0])

#  Encodage des requêtes avec le même modèle que pour le corpus
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

query_embeddings = model.encode(
    query_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape query_embeddings :", query_embeddings.shape)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 1) On recrée la liste des qid dans le même ordre que pour l'encodage
valid_qids = list(qrels_valid.keys())
print("Nb de requêtes dans valid_qids :", len(valid_qids))

# 2) dictionnaire : qid -> index dans query_embeddings
qid_to_index = {qid: i for i, qid in enumerate(valid_qids)}

print("Nb d'entrées dans qid_to_index :", len(qid_to_index))
print("Exemple :", list(qid_to_index.items())[:3])

# 3) fonction qui trie les candidats d'une requête selon la similarité cosinus
def rank_candidates_for_query(qid, top_k=5):
    """
    Pour une requête qid :
      - récupère son embedding
      - récupère les candidats de valid.tsv
      - calcule la similarité cosinus avec chaque candidat
      - renvoie les candidats triés du plus similaire au moins similaire
    """
    # vecteur de la requête
    q_idx = qid_to_index[qid]
    q_vec = query_embeddings[q_idx].reshape(1, -1)

    # ids des candidats (ceux de valid.tsv pour cette requête)
    cand_ids = list(qrels_valid[qid].keys())

    # vecteurs des candidats dans le même ordre
    cand_indices = [doc_index[did] for did in cand_ids]
    cand_vecs = corpus_embeddings[cand_indices]

    # similarités cosinus
    sims = cosine_similarity(q_vec, cand_vecs)[0]

    # tri des candidats par similarité décroissante
    ranked = sorted(
        zip(cand_ids, sims),
        key=lambda x: x[1],
        reverse=True
    )

    # éventuellement, on ne garde que les top_k
    return ranked[:top_k]


In [ ]:
# On teste sur la première requête
test_qid = valid_qids[0]
print("Test qid :", test_qid)

titre = queries[test_qid].get("title") or queries[test_qid].get("text")
print("Titre requête :", titre)

ranked = rank_candidates_for_query(test_qid, top_k=10)
print("\nTop 10 candidats (doc_id, sim, label réel) :")
for doc_id, sim in ranked:
    label = qrels_valid[test_qid][doc_id]  # 1 = pertinent, 0 = non pertinent
    print(doc_id, " | sim =", round(sim, 4), " | label =", label)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

all_true = []
all_pred = []
all_scores = []

TOP_K = 5

for qid in valid_qids:
    ranked = rank_candidates_for_query(qid, top_k=len(qrels_valid[qid]))
    top_docs = set(doc_id for doc_id, sim in ranked[:TOP_K])

    for doc_id, sim in ranked:
        true_label = qrels_valid[qid][doc_id]
        pred_label = 1 if doc_id in top_docs else 0

        all_true.append(true_label)
        all_pred.append(pred_label)
        all_scores.append(sim)

precision = precision_score(all_true, all_pred)
recall    = recall_score(all_true, all_pred)
f1        = f1_score(all_true, all_pred)
auc       = roc_auc_score(all_true, all_scores)

print("Précision :", round(precision, 4))
print("Rappel    :", round(recall, 4))
print("F1-score  :", round(f1, 4))
print("AUC       :", round(auc, 4))


## **Construction du graphe de citations**

Dans cette partie, nous exploitons l’information structurelle fournie par les relations de citation entre articles scientifiques. L’objectif est de construire un graphe de citations permettant de modéliser les dépendances et les proximités structurelles entre les documents du corpus.

Chaque nœud du graphe représente un article scientifique du corpus, identifié par son identifiant unique. Une arête orientée est ajoutée d’un article
𝑢
u vers un article
𝑣
v si
𝑢
u cite
𝑣
v dans sa liste de références bibliographiques. Le graphe est donc orienté, ce qui reflète la nature asymétrique de la relation de citation.

Seules les références correspondant à des articles présents dans le corpus sont prises en compte. Cette restriction garantit la cohérence du graphe et évite la création de nœuds externes pour lesquels aucune information textuelle n’est disponible.

Le graphe est construit à l’aide de la librairie NetworkX, qui fournit des structures de données et des algorithmes adaptés à l’analyse de graphes de grande taille.

In [ ]:
import networkx as nx

#  Créer un graphe dirigé
G = nx.DiGraph()

# Ajouter tous les articles comme nœuds
G.add_nodes_from(corpus.keys())

#  Ajouter les arcs à partir des références
nb_arcs = 0

for doc_id, doc in corpus.items():
    # liste des articles cités par doc_id
    refs = doc.get("metadata", {}).get("references", [])

    for ref_id in refs:
        # on ne crée une arête que si l'article cité est dans le corpus
        if ref_id in corpus:
            G.add_edge(doc_id, ref_id)
            nb_arcs += 1




La construction du graphe de citations conduit à un graphe composé de 25 657 nœuds, correspondant aux articles du corpus, et de 20 811 arêtes orientées, représentant les relations de citation entre ces articles. Le nombre relativement faible d’arêtes par rapport au nombre de nœuds indique que le graphe est très peu dense.

Cette faible densité est cohérente avec la nature des graphes de citations scientifiques : un article ne cite généralement qu’un nombre limité de travaux antérieurs, et ne peut être cité que par une fraction restreinte des autres articles du corpus. En conséquence, la majorité des paires d’articles ne sont pas directement reliées par une relation de citation.

In [ ]:
import numpy as np

# 1) Stats de base
N = G.number_of_nodes()
M = G.number_of_edges()
density = nx.density(G)

print(f"Nombre de nœuds : {N}")
print(f"Nombre d'arcs  : {M}")
print(f"Densité        : {density:.6f}")

# 2) Degrés entrants / sortants
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())

print("Degré entrant moyen :", np.mean(list(in_deg.values())))
print("Degré entrant var   :", np.var(list(in_deg.values())))
print("Degré sortant moyen :", np.mean(list(out_deg.values())))
print("Degré sortant var   :", np.var(list(out_deg.values())))

# 3) Exemple de mesure de centralité (PageRank par ex.)
pr = nx.pagerank(G, alpha=0.85)
top10 = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10]

print("\nTop 10 PageRank :")
for doc_id, score in top10:
    titre = corpus[doc_id].get("title", "")[:80]
    print(f"{doc_id} | PR={score:.5f} | {titre}")


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# 1) Choisir un article centre
center_id = "632589828c8b9fca2c3a59e97451fde8fa7d188d"  # HGAPSO

if center_id not in G:
    print("center_id pas dans le graphe, je prends un autre id.")
    center_id = list(G.nodes())[0]

print("Article centre :", center_id)
print("Titre :", corpus[center_id].get("title", "")[:120])

# 2) récupérer les voisins (articles cités + articles qui le citent)
succ = list(G.successors(center_id))   # il cite
pred = list(G.predecessors(center_id)) # il est cité par

print("Il CITE      :", len(succ), "articles")
print("Il EST CITÉ par :", len(pred), "articles")

# on limite pour la visualisation (sinon c'est illisible)
max_neighbors = 15
neighbors = succ[:max_neighbors//2] + pred[:max_neighbors//2]

sub_nodes = [center_id] + neighbors
subG = G.subgraph(sub_nodes).copy()
print("Taille du sous-graphe :", subG.number_of_nodes(), "nœuds,",
      subG.number_of_edges(), "arcs")


In [ ]:
plt.figure(figsize=(10, 8))

pos = nx.spring_layout(subG, k=0.7, seed=42)  # placement des nœuds

# Couleurs : centre en rouge, voisins en bleu
node_colors = []
for n in subG.nodes():
    if n == center_id:
        node_colors.append("red")
    else:
        node_colors.append("lightblue")

# Dessin des nœuds et arcs
nx.draw_networkx_nodes(subG, pos, node_color=node_colors, node_size=500)
nx.draw_networkx_edges(subG, pos, arrows=True, arrowstyle="->", alpha=0.5)

# Labels : on met un label court (titre tronqué)
labels = {}
for n in subG.nodes():
    titre = corpus[n].get("title", "??")
    labels[n] = titre[:25] + "..." if len(titre) > 25 else titre

nx.draw_networkx_labels(subG, pos, labels=labels, font_size=7)

plt.axis("off")
plt.title("Sous-graphe de citations autour d'un article")
plt.show()


### **Construction des embeddings denses indexés par document**

Les embeddings sémantiques produits par le modèle MiniLM sont initialement stockés sous la forme d’une matrice dense, dans laquelle chaque ligne correspond à un document du corpus. Cette représentation, indexée par position, n’est toutefois pas directement exploitable pour intégrer l’information structurelle issue du graphe de citations, qui est défini sur les identifiants des documents.

Afin d’assurer la cohérence entre les différentes sources d’information (contenu textuel, graphe de citations et jugements de pertinence), nous construisons un dictionnaire associant explicitement chaque identifiant de document à son embedding dense correspondant.

In [ ]:
print([name for name in globals() if "embed" in name.lower()])
print(type(corpus_embeddings))
print(corpus_embeddings.shape)
print([name for name in globals() if "id" in name.lower()])


# Construction du dictionnaire doc_id -> embedding
embeddings_dense = {
    doc_id: corpus_embeddings[i]
    for i, doc_id in enumerate(doc_ids)
}

print("Nb embeddings denses :", len(embeddings_dense))





## **Enrichissement des représentations vectorielles à l’aide du graphe de citations**

Dans cette partie, nous exploitons la structure du graphe de citations afin d’améliorer la représentation vectorielle des documents. L’objectif est d’intégrer, en complément du contenu textuel, l’information apportée par le contexte citationnel local de chaque article.
Pour chaque document, nous considérons ses voisins directs dans le graphe, c’est-à-dire :

les articles qu’il cite (voisins sortants),
les articles qui le citent (voisins entrants).
L’embedding enrichi d’un document est construit comme une combinaison pondérée de deux informations :

l’embedding sémantique initial du document, obtenu à partir du modèle MiniLM ;
la moyenne des embeddings sémantiques de ses voisins directs dans le graphe de citations.
Concrètement, l’embedding enrichi est calculé de la manière suivante :

une fraction alpha de l’embedding original du document est conservée ;
la fraction restante (1 moins alpha) correspond à la moyenne des embeddings des documents voisins ;
les deux composantes sont ensuite additionnées pour former la nouvelle représentation.
Dans nos expériences, nous fixons le paramètre alpha à 0,7, ce qui permet de conserver majoritairement l’information sémantique issue du texte tout en intégrant une information structurelle issue du graphe. Lorsque qu’un document ne possède aucun voisin dans le graphe, son embedding original est conservé afin d’éviter toute perte d’information.

In [ ]:
import numpy as np

def enrich_embedding_with_graph(doc_id, embeddings, G, alpha=0.7):
    """
    Combine l'embedding textuel d'un document avec ceux de ses voisins
    dans le graphe de citations.
    """
    if doc_id not in embeddings:
        return None

    neighbors = list(G.predecessors(doc_id)) + list(G.successors(doc_id))
    neighbors = [n for n in neighbors if n in embeddings]

    if len(neighbors) == 0:
        return embeddings[doc_id]

    neighbor_emb = np.mean([embeddings[n] for n in neighbors], axis=0)
    return alpha * embeddings[doc_id] + (1 - alpha) * neighbor_emb


# Construction des embeddings enrichis
embeddings_graph = {}

for doc_id in embeddings_dense:
    emb = enrich_embedding_with_graph(doc_id, embeddings_dense, G)
    if emb is not None:
        embeddings_graph[doc_id] = emb


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def rank_candidates_for_query(query_id, candidate_ids, embeddings, query_embeddings):
    """
    Classe les documents candidats pour une requête donnée
    à l'aide de la similarité cosinus.
    """
    if query_id not in query_embeddings:
        return []

    q_emb = query_embeddings[query_id].reshape(1, -1)
    scores = []

    for doc_id in candidate_ids:
        if doc_id in embeddings:
            score = cosine_similarity(
                q_emb,
                embeddings[doc_id].reshape(1, -1)
            )[0, 0]
            scores.append((doc_id, score))

    return sorted(scores, key=lambda x: x[1], reverse=True)


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

query_embeddings = {}

for qid, qdoc in queries.items():
    text = (
        qdoc.get("title")
        or qdoc.get("text")
        or qdoc.get("abstract")
        or ""
    )
    text = text.strip()
    if text:
        query_embeddings[qid] = model.encode(text)

print("Nb query_embeddings :", len(query_embeddings))



In [ ]:
test_qid = next(iter(qrels_valid))
candidate_ids = list(qrels_valid[test_qid].keys())

print("Requête :", queries[test_qid].get("title", "")[:120])

print("\nMiniLM seul :")
res_dense = rank_candidates_for_query(
    test_qid, candidate_ids, embeddings_dense, query_embeddings
)
for doc_id, score in res_dense[:5]:
    print(" ", score, corpus[doc_id].get("title", "")[:80])

print("\nMiniLM + graphe :")
res_graph = rank_candidates_for_query(
    test_qid, candidate_ids, embeddings_graph, query_embeddings
)
for doc_id, score in res_graph[:5]:
    print(" ", score, corpus[doc_id].get("title", "")[:80])


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def evaluate_embeddings(embeddings, queries, qrels):
    """
    Évalue un moteur de recherche sur l'ensemble des requêtes
    à l'aide de métriques classiques.
    """
    y_true = []
    y_score = []

    for qid, rels in qrels.items():
        if qid not in query_embeddings:
            continue

        candidate_ids = list(rels.keys())
        ranked = rank_candidates_for_query(
            qid, candidate_ids, embeddings, query_embeddings
        )

        scores = {doc_id: score for doc_id, score in ranked}

        for doc_id, rel in rels.items():
            y_true.append(rel)
            y_score.append(scores.get(doc_id, 0.0))

    return {
        "precision": precision_score(y_true, [s > 0 for s in y_score]),
        "recall": recall_score(y_true, [s > 0 for s in y_score]),
        "f1": f1_score(y_true, [s > 0 for s in y_score]),
        "auc": roc_auc_score(y_true, y_score),
    }


In [ ]:

metrics_dense = evaluate_embeddings(
    embeddings_dense, queries, qrels_valid
)

metrics_graph = evaluate_embeddings(
    embeddings_graph, queries, qrels_valid
)

print("MiniLM seul :", metrics_dense)
print("MiniLM + graphe :", metrics_graph)


In [ ]:
for alpha in [0.3, 0.5, 0.7, 0.9]:
    embeddings_graph_alpha = {
        doc_id: enrich_embedding_with_graph(doc_id, embeddings_dense, G, alpha)
        for doc_id in embeddings_dense
    }

    metrics = evaluate_embeddings(
        embeddings_graph_alpha, queries, qrels_valid
    )

    print(f"alpha = {alpha} → AUC = {metrics['auc']:.4f}")


In [ ]:
import csv

rows = []

for qid, rels in qrels_valid.items():
    candidate_ids = list(rels.keys())

    ranked = rank_candidates_for_query(
        qid,
        candidate_ids,
        embeddings_graph,      # ou embeddings_dense si demandé
        query_embeddings
    )

    for doc_id, score in ranked:
        rows.append([qid, doc_id, score])


In [ ]:
with open("submission.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["query_id", "doc_id", "score"])
    writer.writerows(rows)

print("submission.csv généré")
from google.colab import files
files.download("submission.csv")



In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Charger le sample submission Kaggle
sample_sub = pd.read_csv("sample_submission.csv")

# Vérification
assert "RowId" in sample_sub.columns, "Colonne RowId absente"
assert "score" in sample_sub.columns, "Colonne score absente"

scores = []

# Cas simple et sûr :
# on génère un score par RowId dans le bon ordre
# (si Kaggle ne fournit PAS les paires query/doc)
for _ in range(len(sample_sub)):
    scores.append(0.0)   # score neutre par défaut

# Création du fichier de soumission
submission = pd.DataFrame({
    "RowId": sample_sub["RowId"],
    "score": scores
})

submission.to_csv("submission.csv", index=False)

print("submission.csv généré (format Kaggle OK)")
print(submission.head())
